# Ridge with the shared evaluation

We run Ridge's saved errors (`models/Ridge/ridge_errors.csv`) through Vincent's `calibrate` and `run_evaluation` functions (`models/Transformer/evaluate_recon_model.py`). This way Ridge is scored with the same code as the Transformer. The results go to `models/Ridge/evaluation/`.

## 1. Score Ridge with the shared code

Why: we want Ridge and the Transformer to use the exact same scores, threshold rule and metrics. The shared functions expect bigger = worse. Ridge's errors are signed (real value minus prediction), so they can be negative. So we square them first. Vincent's section 8 (`models/Transformer/scoring_and_metrics.md`) says to use the squared error for a model that predicts values.

What the code does:
- Loads `ridge_errors.csv`, splits it into Validation (`val_errors`), Calibration (`calib_errors`) and Dataset 2 (`dev_errors`), and squares the errors.
- Loads `dataset2_eval.csv` for the labels (`dev_labels`) and attack ids (`dev_attack_id`). The `assert` checks that the errors and `dataset2_eval.csv` are the same hours in the same order, so each label matches the right error.
- Prints the shapes and the number of sensors.
- `calibrate` sets a threshold on Calibration for each of three scores: `topk_percentile`, `mean` and `topk_ratio`. It uses the top 3 sensors, and the budget is at most 1% of hours under alarm and at most 1 alarm episode per week. It also checks the budget on Validation.
- We use the team's shared settings, the same as the Transformer and Isolation Forest: merge gap 2 (`merge_gap_hours=2`: alarm runs separated by 2 or fewer quiet hours count as one episode) and grace hours 3 (`grace_hours=3`: the 3 hours after each attack ends aren't counted).
- `run_evaluation` scores Dataset 2 with these thresholds, prints the results table, and saves the result files and plots in `out_dir`.

Caution: `evaluate_recon_model.py` imports PyTorch. On Windows with Anaconda, this crashed the kernel with "OMP: Error #15" (two copies of the OpenMP library were loaded). As a workaround, we started Jupyter from Anaconda Prompt after running `set KMP_DUPLICATE_LIB_OK=TRUE`. Intel calls this unsafe, because in rare cases it can crash or give wrong results. For this scoring (squaring, sorting, averaging) it is very unlikely to matter. The real fix would be to move the scoring functions into their own file, without the PyTorch import.

In [1]:
import sys
import os
import numpy as np
import pandas as pd

sys.path.append("../models/Transformer")
from evaluate_recon_model import calibrate, run_evaluation

errs = pd.read_csv("../models/Ridge/ridge_errors.csv", parse_dates=["DATETIME"])
sensors = [c for c in errs.columns if c not in ["DATETIME", "split"]]

val_errors = errs.loc[errs["split"] == "validation", sensors].to_numpy() ** 2
calib_errors = errs.loc[errs["split"] == "calibration", sensors].to_numpy() ** 2
dev = errs[errs["split"] == "dataset2"].reset_index(drop=True)
dev_errors = dev[sensors].to_numpy() ** 2

d2 = pd.read_csv("../data/processed/dataset2_eval.csv", parse_dates=["DATETIME"])
assert (d2["DATETIME"].values == dev["DATETIME"].values).all()

dev_when = d2["DATETIME"].to_numpy()
dev_labels = d2["label"].to_numpy()
dev_attack_id = d2["attack_id"].to_numpy()

print(val_errors.shape, calib_errors.shape, dev_errors.shape, len(sensors))

out_dir = "../models/Ridge/evaluation"
os.makedirs(out_dir, exist_ok=True)

calibration = calibrate(calib_errors, val_errors, merge_gap_hours=2)
results = run_evaluation(calibration, dev_errors, dev_when, dev_labels, dev_attack_id,
                         sensors, out_dir=out_dir, grace_hours=3)



(1728, 28) (1729, 28) (4177, 28) 28
####################################################################################################
Results on Dataset 2 (4,177 scored hours: 492 attack, 3,664 clean)
                                topk_percentile     mean                topk_ratio          
Attacks detected                7 of 7              7 of 7              7 of 7              
Missed attacks (Attack ID:)     none                none                none                
Mean delay (h, detected only)   7.7                 6.0                 7.7                 
False-alarm episodes per week   1.38                1.24                1.33                
Clean hours under alarm         0.96%               0.85%               1.01%               
Inside budget on Dataset 2      False               False               False               
Inside budget on validation     False               True                False               
Precision                       0.894               

## 2. Check the saved files

Why: we check that the files were saved, and that `results` holds the same numbers as the printed table.

What the code does: lists the files in `out_dir`, and prints the F1 of `topk_percentile` and `topk_ratio` from `results`.

In [2]:
print(os.listdir(out_dir))
print(results["topk_percentile_f1"], results["topk_ratio_f1"])

['attack_1.png', 'attack_2.png', 'attack_3.png', 'attack_4.png', 'attack_5.png', 'attack_6.png', 'attack_7.png', 'calibration_mean.png', 'calibration_topk_percentile.png', 'calibration_topk_ratio.png', 'detection_delay.png', 'false_alarm_1.png', 'false_alarm_2.png', 'false_alarm_3.png', 'hourly_scores.csv', 'metrics.json', 'per_attack.csv', 'timeline_mean.png', 'timeline_topk_percentile.png', 'timeline_topk_ratio.png', 'top_sensors_per_attack.png']
0.7161997563946407 0.7389749702026223


**What this shows:**
- Validation has 1,728 hours, Calibration 1,729 and Dataset 2 4,177, each with 28 sensors. The `assert` passed, so the hours match.
- Dataset 2 has 492 attack hours and 3,664 clean hours. The 21 grace hours (3 after each of the 7 attacks) are not counted.
- All 7 attacks are detected with all three scores. No attack is missed.
- Our first run used merge gap 0 and grace 0. With 2 and 3, the episodes per week dropped slightly (1.46 to 1.38 for `topk_percentile`), the threshold stayed the same, and Ridge is still outside the budget on Dataset 2.
- Our own Ridge results (`docs/ridge.md`, section 7) next to the shared code:

| | Our Ridge | `topk_ratio` | `topk_percentile` |
|---|---|---|---|
| Precision | 0.893 | 0.893 | 0.894 |
| Recall | 0.610 | 0.630 | 0.598 |
| F1 | 0.725 | 0.739 | 0.716 |
| ROC-AUC | 0.866 | 0.864 | 0.864 |
| PR-AUC | 0.741 | 0.749 | 0.731 |
| False alarm episodes per week | 1.46 | 1.33 | 1.38 |

- `topk_ratio` is closest to our method: both divide each sensor's error by a normal size for that sensor, and average the top 3 sensors. The numbers are almost the same, so both pieces of code agree.
- Our own Ridge counts episodes without a merge gap or grace hours, so its 1.46 per week uses a slightly stricter rule than the other two columns.
- The F1 values printed by the second cell (0.7162 and 0.7390) match the table.
- All three scores are outside the budget on Dataset 2. They all have more than 1 false alarm episode per week (1.38, 1.24 and 1.33). `topk_ratio` also has 1.01% of clean hours under alarm, just over the 1% limit.
- `topk_percentile` and `topk_ratio` are also outside the budget on Validation. Only `mean` stays inside.
- The S score is about 0.84 to 0.85 (0.840, 0.846 and 0.848).
- The `topk_percentile` threshold is 0.9975 (printed as 0.99749). This is close to its ceiling of 1, because a percentile can't go above 1. So there is very little room between the threshold and the highest possible score.

**Ridge vs the Transformer** (both with `topk_percentile` and the same settings: top 3 sensors, merge gap 2, grace hours 3). The Transformer numbers are from `models/Transformer`, the frozen 12-hour model, seed 0.

| | Ridge | Transformer |
|---|---|---|
| Precision | 0.894 | 0.906 |
| Recall | 0.598 | 0.630 |
| F1 | 0.716 | 0.743 |
| ROC-AUC | 0.864 | 0.919 |
| PR-AUC | 0.731 | 0.792 |
| S score | 0.840 | 0.895 |
| Mean delay (hours) | 7.7 | 1.7 |
| False alarm episodes per week | 1.38 | 1.15 |

- The Transformer is better on every measure. The biggest gaps are in ranking (ROC-AUC and PR-AUC) and speed (mean delay). This is likely because the Transformer sees the last 12 hours, while Ridge sees one hour at a time.
- Both are outside the budget on Dataset 2 (more than 1 false alarm episode per week).

- 21 files are saved in `models/Ridge/evaluation/`:
  - `metrics.json`, `per_attack.csv` and `hourly_scores.csv`
  - `calibration_<score>.png` and `timeline_<score>.png`, one of each for `topk_percentile`, `mean` and `topk_ratio`
  - `attack_1.png` to `attack_7.png`, `top_sensors_per_attack.png` and `detection_delay.png`
  - `false_alarm_1.png` to `false_alarm_3.png`

Caution: the competition's S scores are on the Test set. Ours are on Dataset 2, so we can't compare them with the competition yet.

## End of notebook

The short write-up of these results is in `docs/ridge.md`, section 11 ("Ridge with the shared evaluation").